In [1]:
import os

In [4]:
original_folder_books_with_out_edit=r"/mnt/c/Users/arwah/Autism_Chat_bot/src/assets/data/books"
len(os.listdir(original_folder_books_with_out_edit))

10

---

#  Text Extraction & Cleaning Pipeline

This pipeline is designed to extract text from **PDF**, **TXT**, and **EPUB** files, then clean and normalize it for downstream tasks such as embeddings or retrieval‑augmented generation (RAG).

---

## 1. `clean_text(text: str) -> str`
**Purpose:** Final normalization step.  
- Removes page markers like `Page 12` or `- 12 -`.  
- Removes standalone page numbers (lines containing only digits).  
- Normalizes whitespace: collapses multiple spaces, preserves paragraph breaks.  
- Returns a clean, continuous text string.

**Example:**
```text
Page 12
Autism spectrum disorder overview.

13
Symptoms and diagnosis.
```
 After cleaning:
```text
Autism spectrum disorder overview.

Symptoms and diagnosis.
```

---

## 2. `skip_until_chapter(text: str, keywords=("Chapter 1", "Introduction"), skip=True)`
**Purpose:** Skips front‑matter pages (publisher info, acknowledgments, TOC) until the first real chapter.  
- Searches for keywords like `"Chapter 1"` or `"Introduction"`.  
- If found, returns text starting from that point.  
- If not found, returns the full text.

**Example:**
```text
Acknowledgments
Publisher Info
Chapter 1: Autism Spectrum Disorder
```
 After skipping:
```text
Chapter 1: Autism Spectrum Disorder
```

---

## 3. `remove_toc_and_footnotes(text: str) -> str`
**Purpose:** Removes table of contents lines and footnotes.  
- Skips lines with dotted leaders (`.....`).  
- Skips footnote lines starting with symbols (`*`, `†`, `‡`).  
- Skips numbered footnotes like `1. See also...`.

**Example:**
```text
Chapter 1 ..... 12
Chapter 2 ..... 34
1. See Appendix A
† Reference study
Autism spectrum disorder overview...
```
 After cleaning:
```text
Autism spectrum disorder overview...
```

---

## 4. `detect_headings(text: str) -> str`
**Purpose:** Tags chapter headings for downstream chunking.  
- Uses regex to detect headings like `"Chapter 1"`, `"Chapter II"`, etc.  
- Wraps them in `[HEADING]...[/HEADING]` markers.

**Example:**
```text
Chapter 1: Autism Spectrum Disorder
```
 After tagging:
```text
[HEADING]Chapter 1: Autism Spectrum Disorder[/HEADING]
```

---

## 5. `extract_text(file_path: str, skip_front_matter: bool = True) -> str`
**Purpose:** Extracts raw text depending on file type.  
- **PDF:** Uses PyMuPDF (`fitz`) to extract blocks, sorted for correct reading order.  
- **TXT:** Reads plain text directly.  
- **EPUB:** Uses `ebooklib` + `BeautifulSoup` to parse HTML content, stripping scripts, styles, and superscripts.  
- Returns raw text (not yet cleaned).

---

## 6. `cleaning_pipeline(file_path: str, skip_front_matter: bool = True) -> str`
**Purpose:** Orchestrates the full cleaning process.  
- Step 1: Extract raw text (`extract_text`).  
- Step 2: Skip front‑matter (`skip_until_chapter`).  
- Step 3: Remove TOC and footnotes (`remove_toc_and_footnotes`).  
- Step 4: Detect headings (`detect_headings`).  
- Step 5: Final normalization (`clean_text`).  
- Returns fully cleaned text ready for NLP tasks.

---

##  Workflow Summary
1. **Extract raw text** from PDF/TXT/EPUB.  
2. **Skip front‑matter** until first chapter.  
3. **Remove TOC & footnotes** for cleaner flow.  
4. **Tag headings** for structured chunking.  
5. **Normalize text** into clean paragraphs.  

---

##  Output
The final result is a **clean, normalized text corpus**:
- Starts at the first chapter.  
- Free of TOC, footnotes, headers/footers.  
- Structured with heading tags.  
- Ready for embeddings, search, or RAG pipelines.  

---






# clean text 

Page 12

This   is   the   first   paragraph.   

- 13 -

This is   the second paragraph, with   extra spaces.  

45

This is the third one.

### Step 1: Remove page markers like "Page 12" or "- 12 -"
python
text = re.sub(r'\bPage\s+\d+\b', '', text, flags=re.IGNORECASE)
Matches "Page 12" and deletes it.

After this step:

_______________________________________________
This   is   the   first   paragraph.   

- 13 -

This is   the second paragraph, with   extra spaces.  

45

This is the third one.
_______________________________________________

### Step 2: Remove lone page numbers (full line only)
python
text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)
Matches lines that are just numbers (like "45" or "- 13 -" if simplified).

After this step:

_______________________________________________
This   is   the   first   paragraph.   



This is   the second paragraph, with   extra spaces.  



This is the third one.
_______________________________________________
### Step 3: Split into paragraphs by blank lines
python
paragraphs = re.split(r'\n{2,}', text)
Splits wherever there are two or more newlines.

Resulting list:
_______________________________________________
python
[
"This   is   the   first   paragraph.   ",
"This is   the second paragraph, with   extra spaces.  ",
"This is the third one."
]
_______________________________________________

### Step 4: Normalize spacing inside each paragraph and rejoin
python
text = '\n\n'.join(' '.join(p.split()) for p in paragraphs if p.strip())
' '.join(p.split()) collapses multiple spaces into single spaces.

Rejoins paragraphs with exactly two newlines between them.

After this step:
_______________________________________________
This is the first paragraph.

This is the second paragraph, with extra spaces.

This is the third one.
_______________________________________________
## Step 5: Final trim
python
return text.strip()
Removes leading/trailing whitespace.

Final output:

This is the first paragraph.

This is the second paragraph, with extra spaces.

This is the third one.

In [ ]:
import re
def clean_text(text: str) -> str:
    """Final normalization — run LAST in the pipeline."""
    # Remove page markers like "Page 12" or "- 12 -"
    text = re.sub(r'\bPage\s+\d+\b', '', text, flags=re.IGNORECASE)
    # Remove lone page numbers (full line only)
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)
    # Normalize whitespace within paragraphs, preserve paragraph breaks
    paragraphs = re.split(r'\n{2,}', text)
    # uses a regex to split the text wherever there are two or more consecutive newline characters (\n{2,}).
    #  Split text into paragraphs by detecting blank lines.
    text = '\n\n'.join(' '.join(p.split()) for p in paragraphs if p.strip())
    #Line 2: Clean each paragraph’s internal spacing and rejoin them with consistent blank lines.
    return text.strip()

# skip untill chapters & removing tocs and footnotes


In [ ]:
def skip_until_chapter(text: str, keywords=("Chapter 1", "Introduction"), skip=True):
    """Skip front-matter. Set skip=False to keep full text."""
    if not skip:
        return text
    for kw in keywords:
        idx = text.find(kw)
        if idx != -1:
            return text[idx:]
    return text  # fallback



def remove_toc_and_footnotes(text: str) -> str:
    """Must run BEFORE clean_text (needs line structure intact)."""
    cleaned_lines = []
    for line in text.splitlines():
        if re.search(r'\.{3,}', line):       # TOC dots
            continue
        if re.match(r'^\s*[\*†‡]+\s', line): # footnote symbols
            continue
        # Remove footnote lines like "1. See also..." only at line start
        if re.match(r'^\s*\d+\.\s+[a-z]', line):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)

# detect Headings and extract text

In [ ]:
from bs4 import BeautifulSoup
from ebooklib import epub, ITEM_DOCUMENT
import fitz
import re
import os

def detect_headings(text: str) -> str:
    """Tag headings for downstream chunking."""
    text = re.sub(
        r'(?im)^(chapter\s+[\divxlcIVXLC]+[\s\-–:]*.*?)$',
        r'\n\n[HEADING]\1[/HEADING]\n\n',
        text
    )
    return text



def extract_text(file_path: str, skip_front_matter: bool = True) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    raw_text = ""

    if ext == ".pdf":
        doc = fitz.open(file_path)
        for page in doc:
            # Sort blocks for correct reading order
            blocks = page.get_text("blocks")
            blocks_sorted = sorted(blocks, key=lambda b: (round(b[1], -1), b[0]))

           #(x0, y0, x1, y1, text, block_no, ...) for each paragraph 
           #x0 = left coordinate
            #y0 = top coordinate
            #x1 = right coordinate
            #y1 = bottom coordinate
            #So b[0] is the x-coordinate (left), and b[1] is the y-coordinate (top).

        '''
                            b[1] → the y-coordinate (vertical position).
                            round(b[1], -1) rounds it to the nearest 10 pixels.
                            Why? Because text lines may not align perfectly (e.g., 100.2 vs 101.7). Rounding groups them into the same “line” bucket.
                            Purpose: group text that’s roughly on the same line, even if the coordinates differ slightly (e.g., 100 vs 101.7).
                            So text that belongs to the same line is treated as equal in vertical position.
                            b[0] → the x-coordinate (horizontal position).
                            Ensures left-to-right order within the same line.
        '''

        raw_text += "\n".join(b[4] for b in blocks_sorted if b[4].strip()) + "\n\n"

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw_text = f.read()

    elif ext == ".epub":
        book = epub.read_epub(file_path)
        for item in book.get_items():
            if item.get_type() == ITEM_DOCUMENT:         # ✅ fixed
                soup = BeautifulSoup(item.get_content(), "html.parser")
                for tag in soup(["script", "style", "sup"]):  # ✅ strip noise
                    tag.decompose()
                raw_text += soup.get_text(separator="\n") + "\n\n"

    else:
        raise ValueError(f"Unsupported file type: {ext}")

    return raw_text


# full pip line

In [ ]:
def cleaning_pipeline(file_path: str, skip_front_matter: bool = True) -> str:
    raw = extract_text(file_path)
    #  Correct order: line-based ops first, normalize last
    step1 = skip_until_chapter(raw, skip=skip_front_matter)
    step2 = remove_toc_and_footnotes(step1)
    step3 = detect_headings(step2)
    step4 = clean_text(step3)
    return step4

# Chunking process

In [ ]:
import re
import uuid
import tiktoken
from datetime import datetime


# ── helpers ──────────────────────────────────────────────────────────────────

def count_tokens(text: str, model: str = "cl100k_base") -> int:
    enc = tiktoken.get_encoding(model)
    return len(enc.encode(text))
# count number of tokens in the whole text ?


def split_into_sentences(text: str) -> list[str]:
    """Sentence-aware split so chunks never cut mid-sentence."""
    # handles Dr., Mr., e.g., abbreviations before splitting
    text = re.sub(r'\b(Dr|Mr|Mrs|Ms|Prof|Sr|Jr|vs|etc|e\.g|i\.e)\.\s', r'\1<PERIOD> ', text)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.replace('<PERIOD>', '.').strip() for s in sentences if s.strip()]
'''
It replaces the . with <PERIOD> temporarily.

Why? Because otherwise the sentence splitter would mistakenly think the period ends the sentence.


Splits the text into sentences wherever there is:

A period (.), exclamation mark (!), or question mark (?)

Followed by whitespace (\s+)

The (?<=...) is a lookbehind — it ensures the split happens after the punctuation.

Example:
"Dr<PERIOD> Smith went to the hospital. He stayed overnight!"  
→ Splits into:
["Dr<PERIOD> Smith went to the hospital.", "He stayed overnight!"]
#  انا عايزاه يقسم جوة chunk 
# بس مينفعش يقسم كدة علي مستوي text
'''



# ── Step 3: Chunker ───────────────────────────────────────────────────────────

def chunk_segment(
    segment: dict,
    target_tokens: int = 400,
    overlap_tokens: int = 80,
    min_tokens: int = 50,
) -> list[dict]:
    """
    segment: a dictionary, expected to have "content" as the text string.


    Chunk a single structural segment into parent + child chunks.

    Strategy:
      - If segment fits in target_tokens → single chunk (no split needed)
      - Otherwise → sliding window over sentences with overlap
      - Returns both child chunks (for retrieval) and a parent ref (for context)
    """
    text = segment["content"]
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    # ── Single chunk (segment is small enough) ────────────────────────────────
    total_tokens = count_tokens(text)
    if total_tokens <= target_tokens:
        return [_build_chunk(
            sentences=sentences,
            segment=segment,
            chunk_index=0,
            total_chunks=1,
            is_parent=True,
        )]

    # ── Sliding window chunking ───────────────────────────────────────────────
    chunks = []
    current_sentences: list[str] = []
    current_tokens = 0
    chunk_index = 0

    for sentence in sentences:
        sent_tokens = count_tokens(sentence)

        # Sentence alone exceeds target — force it as its own chunk
        if sent_tokens > target_tokens:
            if current_sentences:
                chunks.append((list(current_sentences), chunk_index))
                chunk_index += 1
            chunks.append(([sentence], chunk_index))
            chunk_index += 1
            current_sentences = []
            current_tokens = 0
            continue

        if current_tokens + sent_tokens > target_tokens and current_sentences:
            chunks.append((list(current_sentences), chunk_index))
            chunk_index += 1

            # ── Overlap: carry last N tokens worth of sentences forward ───────
            overlap_sents: list[str] = []
            overlap_total = 0
            for s in reversed(current_sentences):
                s_tok = count_tokens(s)
                if overlap_total + s_tok > overlap_tokens:
                    break
                overlap_sents.insert(0, s)
                overlap_total += s_tok

            current_sentences = overlap_sents
            current_tokens = overlap_total

        current_sentences.append(sentence)
        current_tokens += sent_tokens

    # flush final batch
    if current_sentences:
        chunks.append((current_sentences, chunk_index))

    total_chunks = len(chunks)

    # Build chunk dicts, drop any that are too short
    result = []
    for sents, idx in chunks:
        chunk = _build_chunk(
            sentences=sents,
            segment=segment,
            chunk_index=idx,
            total_chunks=total_chunks,
            is_parent=(total_chunks == 1),
        )
        if chunk["token_count"] >= min_tokens:
            result.append(chunk)

    return result


def _build_chunk(
    sentences: list[str],
    segment: dict,
    chunk_index: int,
    total_chunks: int,
    is_parent: bool,
) -> dict:
    """Assemble a raw chunk dict (metadata added in Step 4)."""
    text = " ".join(sentences)
    return {
        "_sentences": sentences,         # kept for overlap logic; removed in Step 4
        "text": text,
        "token_count": count_tokens(text),
        "chunk_index": chunk_index,
        "total_chunks": total_chunks,
        "is_single_chunk": is_parent,
        # carry segment fields forward for metadata enrichment
        "_segment": segment,
    }


# ── Step 4: Metadata enrichment ───────────────────────────────────────────────

def attach_metadata(
    chunk: dict,
    book_title: str,
    book_author: str = "",
    source_file: str = "",
    language: str = "en",
) -> dict:
    """
    Attach rich, retrieval-ready metadata to a chunk.
    Removes internal '_' fields used only for processing.
    """
    seg = chunk.pop("_segment")
    chunk.pop("_sentences", None)

    heading = seg.get("heading", "")
    parent_heading = seg.get("parent_heading") or ""
    chapter_idx = seg.get("chapter_index", 0)
    section_idx = seg.get("section_index", 0)
    level = seg.get("level", 1)
    chunk_index = chunk["chunk_index"]

    # ── Stable, deterministic ID ──────────────────────────────────────────────
    id_str = f"{book_title}|{heading}|{chapter_idx}|{section_idx}|{chunk_index}"
    chunk_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, id_str))

    # ── Breadcrumb for display & reranking ────────────────────────────────────
    breadcrumb_parts = [book_title]
    if parent_heading:
        breadcrumb_parts.append(parent_heading)
    breadcrumb_parts.append(heading)
    breadcrumb = " > ".join(p for p in breadcrumb_parts if p)

    # ── Context prefix injected into text for embedding ───────────────────────
    # This improves retrieval accuracy significantly (Anthropic contextual retrieval pattern)
    context_prefix = f"[{breadcrumb}]\n\n"
    text_with_context = context_prefix + chunk["text"]

    return {
        # ── Identity ──────────────────────────────────────────────────────────
        "chunk_id":         chunk_id,
        "source_file":      source_file,

        # ── Book-level ────────────────────────────────────────────────────────
        "book_title":       book_title,
        "book_author":      book_author,
        "language":         language,

        # ── Hierarchy ─────────────────────────────────────────────────────────
        "level":            level,          # 1=chapter 2=section 3=subsection
        "chapter_index":    chapter_idx,
        "section_index":    section_idx,
        "heading":          heading,
        "parent_heading":   parent_heading,
        "breadcrumb":       breadcrumb,

        # ── Position ──────────────────────────────────────────────────────────
        "chunk_index":      chunk_index,    # within the segment
        "total_chunks":     chunk["total_chunks"],
        "is_single_chunk":  chunk["is_single_chunk"],
        "prev_chunk_id":    None,           # filled in post-linking below
        "next_chunk_id":    None,

        # ── Content ───────────────────────────────────────────────────────────
        "text":             chunk["text"],              # clean text → for storage
        "text_to_embed":    text_with_context,          # prefixed text → for embedding
        "token_count":      chunk["token_count"],

        # ── Audit ─────────────────────────────────────────────────────────────
        "created_at":       datetime.utcnow().isoformat(),
    }


def link_adjacent_chunks(chunks: list[dict]) -> list[dict]:
    """Fill prev/next chunk IDs for sequential context during retrieval."""
    for i, chunk in enumerate(chunks):
        if i > 0:
            chunk["prev_chunk_id"] = chunks[i - 1]["chunk_id"]
        if i < len(chunks) - 1:
            chunk["next_chunk_id"] = chunks[i + 1]["chunk_id"]
    return chunks


# ── Full pipeline entry point ─────────────────────────────────────────────────

def build_chunks(
    segments: list[dict],
    book_title: str,
    book_author: str = "",
    source_file: str = "",
    language: str = "en",
    target_tokens: int = 400,
    overlap_tokens: int = 80,
    min_tokens: int = 50,
) -> list[dict]:
    """
    Runs Steps 3 + 4 over all structural segments.
    Returns a flat list of metadata-enriched, embedding-ready chunks.
    """
    raw_chunks = []
    for segment in segments:
        raw_chunks.extend(
            chunk_segment(segment, target_tokens, overlap_tokens, min_tokens)
        )

    enriched = [
        attach_metadata(c, book_title, book_author, source_file, language)
        for c in raw_chunks
    ]

    return link_adjacent_chunks(enriched)